**Public archival copy.** Saved cell outputs (including embedded images) and Colab-specific runtime metadata have been removed. Original code and explanatory cell content remain unchanged. Some code refers to Google Drive paths and intermediate research artifacts that are not distributed here. The Fitzpatrick17k image files and the researchers' individual access link are **not included**. See `notebooks/README.md` for access and reproducibility notes.


In [ ]:
# ============================================
# Section 1.0
# Mount Google Drive
# ============================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

In [ ]:
# ============================================
# Section 1.1
# Phase 7 Imports and Paths
# ============================================

from pathlib import Path

import json
import math
import random

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    brier_score_loss
)


ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)


DATA_PATH = (
    ARTIFACT_PATH
    / "dataset_splits"
)


PHASE6_PATH = (
    ARTIFACT_PATH
    / "Phase_6_Domain_Adaptation"
)


PHASE7_PATH = (
    ARTIFACT_PATH
    / "Phase_7_Broader_Fairness_Analysis"
)


PHASE7_CONFIG_PATH = (
    PHASE7_PATH
    / "config"
)


PHASE7_TABLE_PATH = (
    PHASE7_PATH
    / "tables"
)


PHASE7_FIGURE_PATH = (
    PHASE7_PATH
    / "figures"
)


PHASE7_MANIFEST_PATH = (
    PHASE7_PATH
    / "manifests"
)


PHASE7_AUDIT_PATH = (
    PHASE7_PATH
    / "audits"
)


PHASE7_BOOTSTRAP_PATH = (
    PHASE7_PATH
    / "bootstrap"
)


for directory in [

    PHASE7_PATH,
    PHASE7_CONFIG_PATH,
    PHASE7_TABLE_PATH,
    PHASE7_FIGURE_PATH,
    PHASE7_MANIFEST_PATH,
    PHASE7_AUDIT_PATH,
    PHASE7_BOOTSTRAP_PATH

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


GLOBAL_SEED = 42

random.seed(
    GLOBAL_SEED
)

np.random.seed(
    GLOBAL_SEED
)


print(
    "Phase 7 root:",
    PHASE7_PATH
)

print(
    "✓ Phase 7 directories ready."
)

In [ ]:
# ============================================
# Section 2.1
# Verify Frozen Phase 6
# ============================================

PHASE6_COMPLETION_FILE = (

    PHASE6_PATH
    / "manifests"
    / "Phase6_Completion_Manifest.json"
)


assert (
    PHASE6_COMPLETION_FILE.exists()
), (
    "Phase 6 completion manifest not found."
)


with open(
    PHASE6_COMPLETION_FILE,
    "r"
) as f:

    phase6_manifest = json.load(
        f
    )


assert (
    phase6_manifest[
        "Status"
    ]
    ==
    "COMPLETE"
)


assert (
    phase6_manifest[
        "Frozen"
    ]
    is True
)


assert (
    float(
        phase6_manifest[
            "Selected_Lambda"
        ]
    )
    ==
    0.0
)


assert (
    phase6_manifest[
        "Test_Data_Used_For_Selection"
    ]
    is False
)


PHASE7_SELECTED_LAMBDA = 0.0


print(
    "✓ Phase 6 status:",
    phase6_manifest[
        "Status"
    ]
)

print(
    "✓ Phase 6 frozen:",
    phase6_manifest[
        "Frozen"
    ]
)

print(
    "✓ Authoritative Phase 7 lambda:",
    PHASE7_SELECTED_LAMBDA
)

print(
    "\n✓ Phase 7 upstream gate passed."
)

In [ ]:
# ============================================
# Section 3.1
# Load Fixed Fair and Dark Test Splits
# ============================================

FAIR_TEST_FILE = (
    DATA_PATH
    / "fair_test.csv"
)


DARK_TEST_FILE = (
    DATA_PATH
    / "dark_test.csv"
)


assert FAIR_TEST_FILE.exists()
assert DARK_TEST_FILE.exists()


fair_test_df = pd.read_csv(
    FAIR_TEST_FILE
)


dark_test_df = pd.read_csv(
    DARK_TEST_FILE
)


fair_test_df[
    "Evaluation_Domain"
] = "Fair"


dark_test_df[
    "Evaluation_Domain"
] = "Dark"


phase7_test_metadata_df = (
    pd.concat(
        [
            fair_test_df,
            dark_test_df
        ],
        ignore_index=True
    )
)


print(
    "Fair test:",
    len(
        fair_test_df
    )
)

print(
    "Dark test:",
    len(
        dark_test_df
    )
)

print(
    "Combined:",
    len(
        phase7_test_metadata_df
    )
)


assert (
    len(
        fair_test_df
    )
    ==
    487
)


assert (
    len(
        dark_test_df
    )
    ==
    162
)


assert (
    len(
        phase7_test_metadata_df
    )
    ==
    649
)


print(
    "\n✓ Fixed Phase 7 evaluation cohort loaded."
)

In [ ]:
# ============================================
# Section 4.1
# Load Phase 6 Final Prediction-Level Results
# ============================================

PHASE6_PREDICTION_FILE = (

    PHASE6_PATH
    / "predictions"
    / "Phase6_Final_Test_Predictions.csv"
)


assert (
    PHASE6_PREDICTION_FILE.exists()
), (
    "Phase 6 final prediction file not found."
)


phase6_predictions_df = pd.read_csv(
    PHASE6_PREDICTION_FILE
)


print(
    "Prediction rows:",
    len(
        phase6_predictions_df
    )
)


print(
    "\nColumns:"
)

print(
    phase6_predictions_df.columns.tolist()
)


print(
    "\nLambda counts:"
)

print(
    phase6_predictions_df[
        "Lambda"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================
# Section 4.2
# Keep Only Authoritative Lambda=0 Predictions
# ============================================

phase7_predictions_df = (

    phase6_predictions_df[

        phase6_predictions_df[
            "Lambda"
        ]
        ==
        PHASE7_SELECTED_LAMBDA
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


print(
    "Selected prediction rows:",
    len(
        phase7_predictions_df
    )
)


print(
    "Seeds:",
    sorted(
        phase7_predictions_df[
            "Seed"
        ]
        .unique()
        .tolist()
    )
)


assert (
    len(
        phase7_predictions_df
    )
    ==
    649 * 5
)


assert (
    sorted(
        phase7_predictions_df[
            "Seed"
        ]
        .unique()
        .tolist()
    )
    ==
    [
        42,
        43,
        44,
        45,
        46
    ]
)


print(
    "\n✓ Only frozen lambda=0 predictions retained."
)

In [ ]:
# ============================================
# Section 4.3
# Prediction Completeness and Duplication Audit
# ============================================

prediction_count_per_image = (

    phase7_predictions_df

    .groupby(
        "md5hash"
    )[
        "Seed"
    ]

    .nunique()
)


print(
    "Unique test images:",
    prediction_count_per_image.shape[0]
)


print(
    "Minimum seeds/image:",
    prediction_count_per_image.min()
)


print(
    "Maximum seeds/image:",
    prediction_count_per_image.max()
)


duplicate_seed_image_rows = (

    phase7_predictions_df

    .duplicated(
        subset=[
            "md5hash",
            "Seed"
        ]
    )

    .sum()
)


print(
    "Duplicate image-seed rows:",
    duplicate_seed_image_rows
)


assert (
    prediction_count_per_image.shape[0]
    ==
    649
)


assert (
    prediction_count_per_image.min()
    ==
    5
)


assert (
    prediction_count_per_image.max()
    ==
    5
)


assert (
    duplicate_seed_image_rows
    ==
    0
)


print(
    "\n✓ Every test image has exactly five seed-specific predictions."
)

In [ ]:
# ============================================
# Section 4.4
# Merge Prediction-Level Results with Metadata
# ============================================

metadata_columns = [

    "md5hash",
    "fitzpatrick_scale",
    "fitzpatrick_centaur",
    "skin_tone",
    "target",
    "Evaluation_Domain"
]


phase7_metadata_subset_df = (

    phase7_test_metadata_df[
        metadata_columns
    ]

    .drop_duplicates(
        subset=[
            "md5hash"
        ]
    )
)


phase7_analysis_df = (

    phase7_predictions_df

    .merge(

        phase7_metadata_subset_df,

        on="md5hash",

        how="left",

        suffixes=(
            "_prediction",
            "_metadata"
        ),

        validate="many_to_one"
    )
)


print(
    "Merged rows:",
    len(
        phase7_analysis_df
    )
)


missing_metadata = (

    phase7_analysis_df[
        "Evaluation_Domain"
    ]

    .isna()

    .sum()
)


print(
    "Rows missing metadata:",
    missing_metadata
)


assert (
    len(
        phase7_analysis_df
    )
    ==
    3245
)


assert (
    missing_metadata
    ==
    0
)


assert (

    phase7_analysis_df[
        "target_prediction"
    ]
    .astype(int)
    .equals(

        phase7_analysis_df[
            "target_metadata"
        ]
        .astype(int)
    )

)


phase7_analysis_df[
    "target"
] = (

    phase7_analysis_df[
        "target_metadata"
    ]
    .astype(int)
)


phase7_analysis_df.drop(

    columns=[
        "target_prediction",
        "target_metadata"
    ],

    inplace=True
)


print(
    "\n✓ Prediction/metadata merge verified."
)

In [ ]:
# ============================================
# Section 5.1
# Audit Fitzpatrick Scale Availability
# ============================================

image_level_metadata_df = (

    phase7_analysis_df

    .drop_duplicates(
        subset=[
            "md5hash"
        ]
    )

    .copy()
)


print(
    "Unique images:",
    len(
        image_level_metadata_df
    )
)


print(
    "\nRaw Fitzpatrick scale counts:"
)


print(

    image_level_metadata_df[
        "fitzpatrick_scale"
    ]

    .value_counts(
        dropna=False
    )

    .sort_index()
)


print(
    "\nFitzpatrick Centaur counts:"
)


print(

    image_level_metadata_df[
        "fitzpatrick_centaur"
    ]

    .value_counts(
        dropna=False
    )

    .sort_index()
)

In [ ]:
# ============================================
# Section 5.2
# Fitzpatrick × Class Sample-Size Audit
# ============================================

fitz_class_table = pd.crosstab(

    image_level_metadata_df[
        "fitzpatrick_scale"
    ],

    image_level_metadata_df[
        "target"
    ],

    dropna=False
)


fitz_class_table.columns = [
    f"Target_{column}"
    for column in fitz_class_table.columns
]


fitz_class_table[
    "Total"
] = (

    fitz_class_table.sum(
        axis=1
    )
)


display(
    fitz_class_table
)

In [ ]:
# ============================================
# Section 5.3
# Fitzpatrick × Evaluation Domain Audit
# ============================================

fitz_domain_table = pd.crosstab(

    image_level_metadata_df[
        "fitzpatrick_scale"
    ],

    image_level_metadata_df[
        "Evaluation_Domain"
    ],

    dropna=False
)


fitz_domain_table[
    "Total"
] = (

    fitz_domain_table.sum(
        axis=1
    )
)


display(
    fitz_domain_table
)

In [ ]:
# ============================================
# Section 5.4
# Freeze Phase 7 Fairness Analysis Plan
# ============================================

PHASE7_PRIMARY_GROUP_VARIABLE = (
    "Evaluation_Domain"
)


PHASE7_SECONDARY_GROUP_VARIABLE = (
    "fitzpatrick_scale"
)


PHASE7_PRIMARY_GROUPS = [
    "Fair",
    "Dark"
]


PHASE7_FITZPATRICK_GROUPS = [
    1,
    2,
    3,
    4,
    5,
    6
]


PHASE7_FINAL_SEEDS = [
    42,
    43,
    44,
    45,
    46
]


PHASE7_MIN_SUBGROUP_N = 20

PHASE7_MIN_CLASS_N = 10


phase7_analysis_plan = {

    "Phase":
        "Phase 7",

    "Title":
        "Broader Fairness Analysis",

    "Authoritative_Model":
        "Phase 6 selected lambda=0",

    "Selected_Lambda":
        0.0,

    "Model_Selection_Reopened":
        False,

    "Primary_Grouping":
        "Fair versus Dark",

    "Fair_Definition":
        "Fitzpatrick I-III",

    "Dark_Definition":
        "Fitzpatrick IV-VI",

    "Secondary_Grouping":
        "Individual Fitzpatrick scale I-VI",

    "Authoritative_Fitzpatrick_Field":
        "fitzpatrick_scale",

    "Fitzpatrick_Centaur_Primary":
        False,

    "Reason_For_Raw_Fitzpatrick_Field":
        (
            "All 649 fixed test images have Fitzpatrick "
            "scale values I-VI, whereas Fitzpatrick "
            "Centaur contains unavailable values."
        ),

    "Minimum_Subgroup_N":
        PHASE7_MIN_SUBGROUP_N,

    "Minimum_Per_Class_N":
        PHASE7_MIN_CLASS_N,

    "Fitzpatrick_VI_Caution":
        (
            "Fitzpatrick VI contains 25 total images "
            "(11 benign, 14 malignant); estimates will "
            "be retained but interpreted with wider "
            "uncertainty."
        ),

    "Final_Seeds":
        PHASE7_FINAL_SEEDS,

    "Test_Cohort_Size":
        649,

    "Group_Definitions_Frozen_Before_Performance_Analysis":
        True,

    "Frozen":
        True
}


PHASE7_ANALYSIS_PLAN_FILE = (

    PHASE7_CONFIG_PATH
    / "Phase7_Fairness_Analysis_Plan.json"
)


with open(
    PHASE7_ANALYSIS_PLAN_FILE,
    "w"
) as f:

    json.dump(
        phase7_analysis_plan,
        f,
        indent=4
    )


print(
    "✓ Phase 7 fairness-analysis plan frozen."
)

print(
    PHASE7_ANALYSIS_PLAN_FILE
)

In [ ]:
# ============================================
# Section 6.1
# Comprehensive Phase 7 Metric Function
# ============================================

def compute_phase7_metrics(
    dataframe
):

    y_true = (
        dataframe[
            "target"
        ]
        .astype(int)
        .to_numpy()
    )


    y_prob = (
        dataframe[
            "probability"
        ]
        .astype(float)
        .to_numpy()
    )


    y_pred = (
        dataframe[
            "prediction"
        ]
        .astype(int)
        .to_numpy()
    )


    n = len(
        dataframe
    )


    n_benign = int(
        np.sum(
            y_true == 0
        )
    )


    n_malignant = int(
        np.sum(
            y_true == 1
        )
    )


    # ----------------------------------------
    # Confusion matrix
    # ----------------------------------------

    tn, fp, fn, tp = (
        confusion_matrix(

            y_true,
            y_pred,

            labels=[
                0,
                1
            ]

        ).ravel()
    )


    sensitivity = (

        tp
        /
        (
            tp
            +
            fn
        )

        if (
            tp + fn
        ) > 0

        else np.nan
    )


    specificity = (

        tn
        /
        (
            tn
            +
            fp
        )

        if (
            tn + fp
        ) > 0

        else np.nan
    )


    fpr = (

        fp
        /
        (
            fp
            +
            tn
        )

        if (
            fp + tn
        ) > 0

        else np.nan
    )


    fnr = (

        fn
        /
        (
            fn
            +
            tp
        )

        if (
            fn + tp
        ) > 0

        else np.nan
    )


    # ----------------------------------------
    # AUC
    # ----------------------------------------

    if len(
        np.unique(
            y_true
        )
    ) == 2:

        auc = float(
            roc_auc_score(
                y_true,
                y_prob
            )
        )

    else:

        auc = np.nan


    # ----------------------------------------
    # Calibration
    # ----------------------------------------

    brier = float(
        brier_score_loss(
            y_true,
            y_prob
        )
    )


    # ----------------------------------------
    # Predictive entropy
    # ----------------------------------------

    eps = 1e-12


    probabilities_clipped = (
        np.clip(
            y_prob,
            eps,
            1.0 - eps
        )
    )


    entropy = -(

        probabilities_clipped
        *
        np.log(
            probabilities_clipped
        )

        +

        (
            1.0
            -
            probabilities_clipped
        )
        *
        np.log(
            1.0
            -
            probabilities_clipped
        )
    )


    return {

        "N":
            int(
                n
            ),

        "N_Benign":
            n_benign,

        "N_Malignant":
            n_malignant,

        "AUC":
            auc,

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "Precision":
            float(
                precision_score(
                    y_true,
                    y_pred,
                    zero_division=0
                )
            ),

        "Recall":
            float(
                recall_score(
                    y_true,
                    y_pred,
                    zero_division=0
                )
            ),

        "Sensitivity":
            float(
                sensitivity
            ),

        "Specificity":
            float(
                specificity
            ),

        "FPR":
            float(
                fpr
            ),

        "FNR":
            float(
                fnr
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    zero_division=0
                )
            ),

        "Brier":
            brier,

        "Entropy":
            float(
                np.mean(
                    entropy
                )
            )
    }


print(
    "✓ Comprehensive Phase 7 metric function defined."
)

In [ ]:
# ============================================
# Section 6.2
# Fair vs Dark Seed-Specific Metrics
# ============================================

domain_metric_rows = []


for seed in PHASE7_FINAL_SEEDS:

    seed_df = (

        phase7_analysis_df[

            phase7_analysis_df[
                "Seed"
            ]
            ==
            seed
        ]

        .copy()
    )


    for domain_name in [
        "Fair",
        "Dark"
    ]:

        subgroup_df = (

            seed_df[

                seed_df[
                    "Evaluation_Domain"
                ]
                ==
                domain_name
            ]
        )


        metrics = compute_phase7_metrics(
            subgroup_df
        )


        domain_metric_rows.append({

            "Seed":
                int(
                    seed
                ),

            "Domain":
                domain_name,

            **metrics
        })


phase7_domain_metrics_df = (
    pd.DataFrame(
        domain_metric_rows
    )
)


display(
    phase7_domain_metrics_df
)

In [ ]:
# ============================================
# Section 6.3
# Fitzpatrick I-VI Seed-Specific Metrics
# ============================================

fitz_metric_rows = []


for seed in PHASE7_FINAL_SEEDS:

    seed_df = (

        phase7_analysis_df[

            phase7_analysis_df[
                "Seed"
            ]
            ==
            seed
        ]

        .copy()
    )


    for fitz_group in (
        PHASE7_FITZPATRICK_GROUPS
    ):

        subgroup_df = (

            seed_df[

                seed_df[
                    "fitzpatrick_scale"
                ]
                ==
                fitz_group
            ]
        )


        metrics = compute_phase7_metrics(
            subgroup_df
        )


        fitz_metric_rows.append({

            "Seed":
                int(
                    seed
                ),

            "Fitzpatrick":
                int(
                    fitz_group
                ),

            **metrics
        })


phase7_fitz_metrics_df = (
    pd.DataFrame(
        fitz_metric_rows
    )
)


display(
    phase7_fitz_metrics_df
)

In [ ]:
# ============================================
# Section 6.4
# Metric Integrity Audit
# ============================================

assert (
    len(
        phase7_domain_metrics_df
    )
    ==
    10
)


assert (
    len(
        phase7_fitz_metrics_df
    )
    ==
    30
)


for seed in PHASE7_FINAL_SEEDS:

    fair_n = int(

        phase7_domain_metrics_df[

            (
                phase7_domain_metrics_df[
                    "Seed"
                ]
                ==
                seed
            )

            &

            (
                phase7_domain_metrics_df[
                    "Domain"
                ]
                ==
                "Fair"
            )
        ][
            "N"
        ]
        .iloc[0]
    )


    dark_n = int(

        phase7_domain_metrics_df[

            (
                phase7_domain_metrics_df[
                    "Seed"
                ]
                ==
                seed
            )

            &

            (
                phase7_domain_metrics_df[
                    "Domain"
                ]
                ==
                "Dark"
            )
        ][
            "N"
        ]
        .iloc[0]
    )


    assert fair_n == 487
    assert dark_n == 162


expected_fitz_n = {

    1: 112,
    2: 220,
    3: 155,
    4: 91,
    5: 46,
    6: 25
}


for fitz_group, expected_n in (
    expected_fitz_n.items()
):

    observed = (

        phase7_fitz_metrics_df[

            phase7_fitz_metrics_df[
                "Fitzpatrick"
            ]
            ==
            fitz_group

        ][
            "N"
        ]
        .unique()
    )


    assert (
        len(
            observed
        )
        ==
        1
    )


    assert (
        int(
            observed[0]
        )
        ==
        expected_n
    )


print(
    "✓ Fair/Dark metric table verified."
)

print(
    "✓ Fitzpatrick I-VI metric table verified."
)

print(
    "✓ All subgroup sample sizes match frozen cohort."
)

In [ ]:
# ============================================
# Section 7.1
# Fair vs Dark Descriptive Summary Across Seeds
# ============================================

DOMAIN_SUMMARY_METRICS = [

    "AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "FPR",
    "FNR",
    "F1",
    "Brier",
    "Entropy"
]


domain_summary_rows = []


for domain_name in [
    "Fair",
    "Dark"
]:

    subset = (

        phase7_domain_metrics_df[

            phase7_domain_metrics_df[
                "Domain"
            ]
            ==
            domain_name
        ]
    )


    row = {

        "Domain":
            domain_name,

        "N":
            int(
                subset[
                    "N"
                ]
                .iloc[0]
            )
    }


    for metric in (
        DOMAIN_SUMMARY_METRICS
    ):

        values = (

            subset[
                metric
            ]
            .to_numpy(
                dtype=float
            )
        )


        row[
            f"{metric}_Mean"
        ] = float(
            np.mean(
                values
            )
        )


        row[
            f"{metric}_SD"
        ] = float(
            np.std(
                values,
                ddof=1
            )
        )


    domain_summary_rows.append(
        row
    )


phase7_domain_summary_df = (
    pd.DataFrame(
        domain_summary_rows
    )
)


display(
    phase7_domain_summary_df.T
)

In [ ]:
# ============================================
# Section 7.2
# Fitzpatrick I-VI Descriptive Summary
# Across Seeds
# ============================================

fitz_summary_rows = []


for fitz_group in (
    PHASE7_FITZPATRICK_GROUPS
):

    subset = (

        phase7_fitz_metrics_df[

            phase7_fitz_metrics_df[
                "Fitzpatrick"
            ]
            ==
            fitz_group
        ]
    )


    row = {

        "Fitzpatrick":
            int(
                fitz_group
            ),

        "N":
            int(
                subset[
                    "N"
                ]
                .iloc[0]
            ),

        "N_Benign":
            int(
                subset[
                    "N_Benign"
                ]
                .iloc[0]
            ),

        "N_Malignant":
            int(
                subset[
                    "N_Malignant"
                ]
                .iloc[0]
            )
    }


    for metric in (
        DOMAIN_SUMMARY_METRICS
    ):

        values = (

            subset[
                metric
            ]
            .to_numpy(
                dtype=float
            )
        )


        row[
            f"{metric}_Mean"
        ] = float(
            np.mean(
                values
            )
        )


        row[
            f"{metric}_SD"
        ] = float(
            np.std(
                values,
                ddof=1
            )
        )


    fitz_summary_rows.append(
        row
    )


phase7_fitz_summary_df = (
    pd.DataFrame(
        fitz_summary_rows
    )
)


display(
    phase7_fitz_summary_df
)

In [ ]:
# ============================================
# Section 8.1
# Freeze Phase 7 Confirmatory Fairness Outcomes
# ============================================

PHASE7_CONFIRMATORY_METRICS = [

    "AUC",

    "Recall",

    "FPR",

    "Brier",

    "Entropy"
]


PHASE7_DESCRIPTIVE_METRICS = [

    "Accuracy",

    "Precision",

    "Specificity",

    "FNR",

    "F1"
]


phase7_outcome_plan = {

    "Primary_Comparison":
        "Fair versus Dark",

    "Confirmatory_Metrics":
        PHASE7_CONFIRMATORY_METRICS,

    "Descriptive_Metrics":
        PHASE7_DESCRIPTIVE_METRICS,

    "Directional_Difference":
        "Dark minus Fair",

    "Multiplicity_Correction":
        "Holm",

    "Alpha":
        0.05,

    "Statistical_Unit":
        "Seed-specific trained model",

    "Number_Seeds":
        5,

    "Paired_Test":
        "Exact two-sided Wilcoxon signed-rank",

    "Fitzpatrick_I_VI":
        (
            "Secondary subgroup analysis; "
            "interpreted with sample-size-aware uncertainty."
        ),

    "FNR_Not_Separately_Confirmatory":
        (
            "FNR is mathematically redundant with "
            "malignant recall."
        ),

    "Frozen_Before_Inferential_Testing":
        True
}


PHASE7_OUTCOME_PLAN_FILE = (

    PHASE7_CONFIG_PATH
    / "Phase7_Confirmatory_Fairness_Outcomes.json"
)


with open(
    PHASE7_OUTCOME_PLAN_FILE,
    "w"
) as f:

    json.dump(
        phase7_outcome_plan,
        f,
        indent=4
    )


print(
    "✓ Phase 7 confirmatory outcomes frozen."
)

print(
    PHASE7_CONFIRMATORY_METRICS
)

In [ ]:
# ============================================
# Section 8.2
# Seed-Level Fair-Dark Disparity Table
# ============================================

ALL_FAIRNESS_METRICS = [

    "AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "FPR",
    "FNR",
    "F1",
    "Brier",
    "Entropy"
]


domain_gap_rows = []


for seed in PHASE7_FINAL_SEEDS:

    fair_row = (

        phase7_domain_metrics_df[

            (
                phase7_domain_metrics_df[
                    "Seed"
                ]
                ==
                seed
            )

            &

            (
                phase7_domain_metrics_df[
                    "Domain"
                ]
                ==
                "Fair"
            )
        ]

        .iloc[0]
    )


    dark_row = (

        phase7_domain_metrics_df[

            (
                phase7_domain_metrics_df[
                    "Seed"
                ]
                ==
                seed
            )

            &

            (
                phase7_domain_metrics_df[
                    "Domain"
                ]
                ==
                "Dark"
            )
        ]

        .iloc[0]
    )


    row = {

        "Seed":
            int(
                seed
            )
    }


    for metric in ALL_FAIRNESS_METRICS:

        fair_value = float(
            fair_row[
                metric
            ]
        )


        dark_value = float(
            dark_row[
                metric
            ]
        )


        directional_difference = (
            dark_value
            -
            fair_value
        )


        row[
            f"Fair_{metric}"
        ] = fair_value


        row[
            f"Dark_{metric}"
        ] = dark_value


        row[
            f"{metric}_Delta_Dark_minus_Fair"
        ] = directional_difference


        row[
            f"{metric}_Absolute_Gap"
        ] = abs(
            directional_difference
        )


    # Equalized-odds style descriptive disparity
    row[
        "Equalized_Odds_Gap"
    ] = max(

        row[
            "Recall_Absolute_Gap"
        ],

        row[
            "FPR_Absolute_Gap"
        ]
    )


    domain_gap_rows.append(
        row
    )


phase7_domain_gap_df = pd.DataFrame(
    domain_gap_rows
)


display(
    phase7_domain_gap_df
)

In [ ]:
# ============================================
# Section 8.3
# Fair-Dark Gap Summary Across Seeds
# ============================================

gap_summary_rows = []


for metric in ALL_FAIRNESS_METRICS:

    directional_values = (

        phase7_domain_gap_df[
            f"{metric}_Delta_Dark_minus_Fair"
        ]
        .to_numpy(
            dtype=float
        )
    )


    absolute_values = (

        phase7_domain_gap_df[
            f"{metric}_Absolute_Gap"
        ]
        .to_numpy(
            dtype=float
        )
    )


    gap_summary_rows.append({

        "Metric":
            metric,

        "Directional_Delta_Mean":
            float(
                np.mean(
                    directional_values
                )
            ),

        "Directional_Delta_SD":
            float(
                np.std(
                    directional_values,
                    ddof=1
                )
            ),

        "Absolute_Gap_Mean":
            float(
                np.mean(
                    absolute_values
                )
            ),

        "Absolute_Gap_SD":
            float(
                np.std(
                    absolute_values,
                    ddof=1
                )
            )
    })


phase7_gap_summary_df = pd.DataFrame(
    gap_summary_rows
)


display(
    phase7_gap_summary_df
)

In [ ]:
# ============================================
# Section 9.1
# Exact Paired Fair-vs-Dark Wilcoxon Tests
# ============================================

from scipy.stats import wilcoxon


fairness_test_rows = []


for metric in (
    PHASE7_CONFIRMATORY_METRICS
):

    fair_values = (

        phase7_domain_metrics_df[

            phase7_domain_metrics_df[
                "Domain"
            ]
            ==
            "Fair"

        ]

        .sort_values(
            "Seed"
        )[
            metric
        ]

        .to_numpy(
            dtype=float
        )
    )


    dark_values = (

        phase7_domain_metrics_df[

            phase7_domain_metrics_df[
                "Domain"
            ]
            ==
            "Dark"

        ]

        .sort_values(
            "Seed"
        )[
            metric
        ]

        .to_numpy(
            dtype=float
        )
    )


    differences = (
        dark_values
        -
        fair_values
    )


    if np.allclose(
        differences,
        0.0
    ):

        statistic = 0.0
        p_value = 1.0

    else:

        result = wilcoxon(

            dark_values,
            fair_values,

            alternative="two-sided",

            method="exact"
        )


        statistic = float(
            result.statistic
        )


        p_value = float(
            result.pvalue
        )


    fairness_test_rows.append({

        "Metric":
            metric,

        "Fair_Mean":
            float(
                np.mean(
                    fair_values
                )
            ),

        "Dark_Mean":
            float(
                np.mean(
                    dark_values
                )
            ),

        "Mean_Delta_Dark_minus_Fair":
            float(
                np.mean(
                    differences
                )
            ),

        "Median_Delta_Dark_minus_Fair":
            float(
                np.median(
                    differences
                )
            ),

        "Wilcoxon_W":
            statistic,

        "Exact_P":
            p_value
    })


phase7_fairness_test_df = (
    pd.DataFrame(
        fairness_test_rows
    )
)


display(
    phase7_fairness_test_df
)

In [ ]:
# ============================================
# Section 9.2
# Holm Correction for Confirmatory Fairness Family
# ============================================

from statsmodels.stats.multitest import (
    multipletests
)


raw_p_values = (

    phase7_fairness_test_df[
        "Exact_P"
    ]

    .to_numpy(
        dtype=float
    )
)


reject, holm_p, _, _ = multipletests(

    raw_p_values,

    alpha=0.05,

    method="holm"
)


phase7_fairness_test_df[
    "Holm_P"
] = holm_p


phase7_fairness_test_df[
    "Holm_Supported"
] = reject


display(
    phase7_fairness_test_df
)

In [ ]:
# ============================================
# Section 10.1
# Hierarchical Fair-Dark Bootstrap
# ============================================

BOOTSTRAP_REPLICATES = 2000

BOOTSTRAP_SEED = 2026


BOOTSTRAP_METRICS = [

    "AUC",
    "Recall",
    "FPR",
    "Brier",
    "Entropy"
]


rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


bootstrap_rows = []


for bootstrap_id in range(
    BOOTSTRAP_REPLICATES
):

    sampled_seeds = rng.choice(

        PHASE7_FINAL_SEEDS,

        size=len(
            PHASE7_FINAL_SEEDS
        ),

        replace=True
    )


    replicate_fair_metrics = {
        metric: []
        for metric
        in BOOTSTRAP_METRICS
    }


    replicate_dark_metrics = {
        metric: []
        for metric
        in BOOTSTRAP_METRICS
    }


    for sampled_seed in sampled_seeds:

        seed_df = (

            phase7_analysis_df[

                phase7_analysis_df[
                    "Seed"
                ]
                ==
                sampled_seed

            ]
        )


        for domain_name in [
            "Fair",
            "Dark"
        ]:

            domain_df = (

                seed_df[

                    seed_df[
                        "Evaluation_Domain"
                    ]
                    ==
                    domain_name
                ]

                .reset_index(
                    drop=True
                )
            )


            sampled_indices = rng.integers(

                low=0,

                high=len(
                    domain_df
                ),

                size=len(
                    domain_df
                )
            )


            sampled_df = (

                domain_df
                .iloc[
                    sampled_indices
                ]
            )


            metrics = compute_phase7_metrics(
                sampled_df
            )


            target_dictionary = (

                replicate_fair_metrics

                if domain_name
                ==
                "Fair"

                else replicate_dark_metrics
            )


            for metric in (
                BOOTSTRAP_METRICS
            ):

                target_dictionary[
                    metric
                ].append(
                    metrics[
                        metric
                    ]
                )


    row = {

        "Bootstrap":
            bootstrap_id
    }


    for metric in BOOTSTRAP_METRICS:

        fair_mean = float(
            np.nanmean(
                replicate_fair_metrics[
                    metric
                ]
            )
        )


        dark_mean = float(
            np.nanmean(
                replicate_dark_metrics[
                    metric
                ]
            )
        )


        difference = (
            dark_mean
            -
            fair_mean
        )


        row[
            f"Fair_{metric}"
        ] = fair_mean


        row[
            f"Dark_{metric}"
        ] = dark_mean


        row[
            f"{metric}_Delta_Dark_minus_Fair"
        ] = difference


        row[
            f"{metric}_Absolute_Gap"
        ] = abs(
            difference
        )


    bootstrap_rows.append(
        row
    )


phase7_bootstrap_df = pd.DataFrame(
    bootstrap_rows
)


print(
    "Bootstrap replicates:",
    len(
        phase7_bootstrap_df
    )
)


print(
    "✓ Hierarchical bootstrap complete."
)

In [ ]:
# ============================================
# Section 10.2
# Bootstrap 95% Confidence Intervals
# ============================================

bootstrap_ci_rows = []


for metric in BOOTSTRAP_METRICS:

    delta_values = (

        phase7_bootstrap_df[
            f"{metric}_Delta_Dark_minus_Fair"
        ]

        .dropna()

        .to_numpy(
            dtype=float
        )
    )


    gap_values = (

        phase7_bootstrap_df[
            f"{metric}_Absolute_Gap"
        ]

        .dropna()

        .to_numpy(
            dtype=float
        )
    )


    bootstrap_ci_rows.append({

        "Metric":
            metric,

        "Delta_Dark_minus_Fair_Lower95":
            float(
                np.percentile(
                    delta_values,
                    2.5
                )
            ),

        "Delta_Dark_minus_Fair_Median":
            float(
                np.percentile(
                    delta_values,
                    50
                )
            ),

        "Delta_Dark_minus_Fair_Upper95":
            float(
                np.percentile(
                    delta_values,
                    97.5
                )
            ),

        "Absolute_Gap_Lower95":
            float(
                np.percentile(
                    gap_values,
                    2.5
                )
            ),

        "Absolute_Gap_Median":
            float(
                np.percentile(
                    gap_values,
                    50
                )
            ),

        "Absolute_Gap_Upper95":
            float(
                np.percentile(
                    gap_values,
                    97.5
                )
            )
    })


phase7_bootstrap_ci_df = (
    pd.DataFrame(
        bootstrap_ci_rows
    )
)


display(
    phase7_bootstrap_ci_df
)

In [ ]:
# ============================================
# Section 10.3
# Save Phase 7 Domain Fairness Results
# ============================================

phase7_domain_metrics_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Domain_Seed_Metrics.csv",

    index=False
)


phase7_domain_gap_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Domain_Seed_Disparities.csv",

    index=False
)


phase7_gap_summary_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Domain_Disparity_Summary.csv",

    index=False
)


phase7_fairness_test_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Confirmatory_Fairness_Tests.csv",

    index=False
)


phase7_bootstrap_ci_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Domain_Bootstrap_CIs.csv",

    index=False
)


phase7_fitz_metrics_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Fitzpatrick_Seed_Metrics.csv",

    index=False
)


phase7_fitz_summary_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Fitzpatrick_Descriptive_Summary.csv",

    index=False
)


print(
    "✓ Current Phase 7 results saved."
)

In [ ]:
# ============================================
# Section 11.1
# Freeze Fitzpatrick I-VI Inferential Plan
# ============================================

PHASE7_FITZ_CONFIRMATORY_METRICS = [

    "AUC",
    "Recall",
    "FPR",
    "Brier",
    "Entropy"
]


phase7_fitz_inferential_plan = {

    "Analysis":
        "Fitzpatrick I-VI secondary fairness analysis",

    "Groups":
        [
            1,
            2,
            3,
            4,
            5,
            6
        ],

    "Metrics":
        PHASE7_FITZ_CONFIRMATORY_METRICS,

    "Independent_Experimental_Unit":
        "Seed-specific trained model",

    "Repeated_Measures_Block":
        "Training seed",

    "Omnibus_Test":
        "Friedman test",

    "Multiplicity_Correction":
        "Holm across five omnibus outcomes",

    "PostHoc_Rule":
        (
            "Pairwise Fitzpatrick comparisons will "
            "only be treated as inferential follow-up "
            "for an omnibus outcome supported after "
            "Holm correction."
        ),

    "Fitzpatrick_VI_Caution":
        (
            "Fitzpatrick VI has only 25 images "
            "(11 benign and 14 malignant), so its "
            "estimates require cautious interpretation."
        ),

    "Frozen_Before_Fitzpatrick_Inferential_Testing":
        True
}


PHASE7_FITZ_PLAN_FILE = (

    PHASE7_CONFIG_PATH
    / "Phase7_Fitzpatrick_Inferential_Plan.json"
)


with open(
    PHASE7_FITZ_PLAN_FILE,
    "w"
) as f:

    json.dump(
        phase7_fitz_inferential_plan,
        f,
        indent=4
    )


print(
    "✓ Fitzpatrick inferential plan frozen."
)

print(
    PHASE7_FITZ_PLAN_FILE
)

In [ ]:
# ============================================
# Section 11.2
# Build Seed-by-Fitzpatrick Metric Matrices
# ============================================

phase7_fitz_matrices = {}


for metric in (
    PHASE7_FITZ_CONFIRMATORY_METRICS
):

    matrix = (

        phase7_fitz_metrics_df

        .pivot(

            index="Seed",

            columns="Fitzpatrick",

            values=metric
        )

        .sort_index()

        .reindex(
            columns=[
                1,
                2,
                3,
                4,
                5,
                6
            ]
        )
    )


    assert (
        matrix.shape
        ==
        (
            5,
            6
        )
    )


    assert (
        matrix.isna()
        .sum()
        .sum()
        ==
        0
    )


    phase7_fitz_matrices[
        metric
    ] = matrix


    print(
        f"\n{metric}"
    )

    display(
        matrix
    )


print(
    "\n✓ All five Fitzpatrick matrices verified."
)

In [ ]:
# ============================================
# Section 11.3
# Fitzpatrick I-VI Friedman Omnibus Tests
# ============================================

from scipy.stats import (
    friedmanchisquare
)


fitz_friedman_rows = []


for metric in (
    PHASE7_FITZ_CONFIRMATORY_METRICS
):

    matrix = (
        phase7_fitz_matrices[
            metric
        ]
    )


    group_arrays = [

        matrix[
            fitz_group
        ]
        .to_numpy(
            dtype=float
        )

        for fitz_group in [
            1,
            2,
            3,
            4,
            5,
            6
        ]
    ]


    result = friedmanchisquare(
        *group_arrays
    )


    fitz_friedman_rows.append({

        "Metric":
            metric,

        "Friedman_Chi2":
            float(
                result.statistic
            ),

        "df":
            5,

        "P_Value":
            float(
                result.pvalue
            )
    })


phase7_fitz_friedman_df = (
    pd.DataFrame(
        fitz_friedman_rows
    )
)


display(
    phase7_fitz_friedman_df
)

In [ ]:
# ============================================
# Section 11.4
# Holm Correction for Fitzpatrick Omnibus Family
# ============================================

fitz_raw_p = (

    phase7_fitz_friedman_df[
        "P_Value"
    ]
    .to_numpy(
        dtype=float
    )
)


fitz_reject, fitz_holm_p, _, _ = (
    multipletests(

        fitz_raw_p,

        alpha=0.05,

        method="holm"
    )
)


phase7_fitz_friedman_df[
    "Holm_P"
] = fitz_holm_p


phase7_fitz_friedman_df[
    "Holm_Supported"
] = fitz_reject


display(
    phase7_fitz_friedman_df
)

In [ ]:
# ============================================
# Section 11.5
# Fitzpatrick Performance Range Summary
# ============================================

fitz_range_rows = []


for metric in (
    PHASE7_FITZ_CONFIRMATORY_METRICS
):

    mean_column = (
        f"{metric}_Mean"
    )


    metric_df = (

        phase7_fitz_summary_df[
            [
                "Fitzpatrick",
                "N",
                mean_column
            ]
        ]

        .copy()
    )


    min_index = (
        metric_df[
            mean_column
        ]
        .idxmin()
    )


    max_index = (
        metric_df[
            mean_column
        ]
        .idxmax()
    )


    minimum_row = (
        metric_df.loc[
            min_index
        ]
    )


    maximum_row = (
        metric_df.loc[
            max_index
        ]
    )


    fitz_range_rows.append({

        "Metric":
            metric,

        "Minimum_Fitzpatrick":
            int(
                minimum_row[
                    "Fitzpatrick"
                ]
            ),

        "Minimum_Mean":
            float(
                minimum_row[
                    mean_column
                ]
            ),

        "Maximum_Fitzpatrick":
            int(
                maximum_row[
                    "Fitzpatrick"
                ]
            ),

        "Maximum_Mean":
            float(
                maximum_row[
                    mean_column
                ]
            ),

        "Range_Max_minus_Min":
            float(
                maximum_row[
                    mean_column
                ]
                -
                minimum_row[
                    mean_column
                ]
            )
    })


phase7_fitz_range_df = (
    pd.DataFrame(
        fitz_range_rows
    )
)


display(
    phase7_fitz_range_df
)

In [ ]:
# ============================================
# Section 11.6
# Save Fitzpatrick Inferential Results
# ============================================

phase7_fitz_friedman_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Fitzpatrick_Friedman_Holm.csv",

    index=False
)


phase7_fitz_range_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Fitzpatrick_Performance_Ranges.csv",

    index=False
)


print(
    "✓ Fitzpatrick inferential tables saved."
)

In [ ]:
# ============================================
# Section 12.1
# Pairwise Fitzpatrick Entropy Post-hoc Tests
# ============================================

from itertools import combinations
from scipy.stats import wilcoxon


entropy_matrix = (
    phase7_fitz_matrices[
        "Entropy"
    ]
)


entropy_pairwise_rows = []


for group_a, group_b in combinations(
    PHASE7_FITZPATRICK_GROUPS,
    2
):

    values_a = (

        entropy_matrix[
            group_a
        ]
        .to_numpy(
            dtype=float
        )
    )


    values_b = (

        entropy_matrix[
            group_b
        ]
        .to_numpy(
            dtype=float
        )
    )


    differences = (
        values_b
        -
        values_a
    )


    if np.allclose(
        differences,
        0.0
    ):

        statistic = 0.0
        p_value = 1.0

    else:

        result = wilcoxon(

            values_b,
            values_a,

            alternative="two-sided",

            method="exact"
        )


        statistic = float(
            result.statistic
        )


        p_value = float(
            result.pvalue
        )


    entropy_pairwise_rows.append({

        "Fitzpatrick_A":
            int(
                group_a
            ),

        "Fitzpatrick_B":
            int(
                group_b
            ),

        "Entropy_A_Mean":
            float(
                np.mean(
                    values_a
                )
            ),

        "Entropy_B_Mean":
            float(
                np.mean(
                    values_b
                )
            ),

        "Mean_Delta_B_minus_A":
            float(
                np.mean(
                    differences
                )
            ),

        "Median_Delta_B_minus_A":
            float(
                np.median(
                    differences
                )
            ),

        "Wilcoxon_W":
            statistic,

        "Exact_P":
            p_value
    })


phase7_entropy_posthoc_df = pd.DataFrame(
    entropy_pairwise_rows
)


display(
    phase7_entropy_posthoc_df
)

In [ ]:
# ============================================
# Section 12.2
# Holm Correction for Entropy Post-hoc Family
# ============================================

entropy_raw_p = (

    phase7_entropy_posthoc_df[
        "Exact_P"
    ]

    .to_numpy(
        dtype=float
    )
)


(
    entropy_reject,
    entropy_holm_p,
    _,
    _

) = multipletests(

    entropy_raw_p,

    alpha=0.05,

    method="holm"
)


phase7_entropy_posthoc_df[
    "Holm_P"
] = entropy_holm_p


phase7_entropy_posthoc_df[
    "Holm_Supported"
] = entropy_reject


phase7_entropy_posthoc_df = (

    phase7_entropy_posthoc_df

    .sort_values(
        [
            "Holm_P",
            "Exact_P"
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    phase7_entropy_posthoc_df
)

In [ ]:
# ============================================
# Section 12.3
# Fitzpatrick Entropy Ordering
# ============================================

phase7_entropy_summary_df = (

    phase7_fitz_summary_df[
        [
            "Fitzpatrick",
            "N",
            "N_Benign",
            "N_Malignant",
            "Entropy_Mean",
            "Entropy_SD"
        ]
    ]

    .sort_values(
        "Entropy_Mean",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


display(
    phase7_entropy_summary_df
)

In [ ]:
# ============================================
# Section 12.4
# Save Entropy Post-hoc Results
# ============================================

phase7_entropy_posthoc_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Fitzpatrick_Entropy_Posthoc_Holm.csv",

    index=False
)


phase7_entropy_summary_df.to_csv(

    PHASE7_TABLE_PATH
    / "Phase7_Fitzpatrick_Entropy_Summary.csv",

    index=False
)


print(
    "✓ Fitzpatrick entropy post-hoc results saved."
)

In [ ]:
# ============================================
# Section 13.1
# Final Phase 7 Result Summary
# ============================================

phase7_final_summary = {

    "Primary_Fair_Dark_Confirmatory_Outcomes":
        5,

    "Primary_Fair_Dark_Holm_Supported":
        int(
            phase7_fairness_test_df[
                "Holm_Supported"
            ]
            .sum()
        ),

    "Fitzpatrick_Omnibus_Outcomes":
        5,

    "Fitzpatrick_Omnibus_Holm_Supported":
        int(
            phase7_fitz_friedman_df[
                "Holm_Supported"
            ]
            .sum()
        ),

    "Supported_Fitzpatrick_Omnibus_Metrics":

        phase7_fitz_friedman_df.loc[
            phase7_fitz_friedman_df[
                "Holm_Supported"
            ],
            "Metric"
        ]
        .tolist(),

    "Entropy_Posthoc_Comparisons":
        int(
            len(
                phase7_entropy_posthoc_df
            )
        ),

    "Entropy_Posthoc_Holm_Supported":
        int(
            phase7_entropy_posthoc_df[
                "Holm_Supported"
            ]
            .sum()
        ),

    "Highest_Entropy_Fitzpatrick":
        int(
            phase7_entropy_summary_df[
                "Fitzpatrick"
            ]
            .iloc[0]
        ),

    "Highest_Entropy_Mean":
        float(
            phase7_entropy_summary_df[
                "Entropy_Mean"
            ]
            .iloc[0]
        ),

    "Lowest_Entropy_Fitzpatrick":
        int(
            phase7_entropy_summary_df[
                "Fitzpatrick"
            ]
            .iloc[-1]
        ),

    "Lowest_Entropy_Mean":
        float(
            phase7_entropy_summary_df[
                "Entropy_Mean"
            ]
            .iloc[-1]
        )
}


for key, value in (
    phase7_final_summary.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================
# Section 13.2
# Create Phase 7 Completion Manifest
# ============================================

phase7_completion_manifest = {

    "Phase":
        "Phase 7",

    "Title":
        "Broader Fairness Analysis",

    "Authoritative_Upstream_Model":
        "Phase 6 selected lambda=0",

    "Selected_Lambda":
        0.0,

    "Model_Retraining_Performed":
        False,

    "Model_Selection_Reopened":
        False,

    "Final_Seeds":
        [
            42,
            43,
            44,
            45,
            46
        ],

    "Evaluation_Cohort_Size":
        649,

    "Fair_Test_Size":
        487,

    "Dark_Test_Size":
        162,

    "Primary_Fairness_Comparison":
        "Fair (Fitzpatrick I-III) versus Dark (IV-VI)",

    "Primary_Confirmatory_Metrics":
        [
            "AUC",
            "Recall",
            "FPR",
            "Brier",
            "Entropy"
        ],

    "Primary_Statistical_Test":
        "Exact paired two-sided Wilcoxon signed-rank",

    "Primary_Multiplicity_Correction":
        "Holm",

    "Primary_Holm_Supported_Outcomes":
        [],

    "Primary_Number_Holm_Supported_Outcomes":
        0,

    "Fair_AUC_Mean":
        0.827458,

    "Dark_AUC_Mean":
        0.876408,

    "Fair_Recall_Mean":
        0.768548,

    "Dark_Recall_Mean":
        0.797368,

    "Fair_FPR_Mean":
        0.264435,

    "Dark_FPR_Mean":
        0.223256,

    "Fair_Brier_Mean":
        0.201716,

    "Dark_Brier_Mean":
        0.167769,

    "Fair_Entropy_Mean":
        0.208701,

    "Dark_Entropy_Mean":
        0.210304,

    "Bootstrap_AUC_Delta_Dark_minus_Fair_95CI":
        [
            0.017157,
            0.080884
        ],

    "Bootstrap_Brier_Delta_Dark_minus_Fair_95CI":
        [
            -0.061159,
            -0.007843
        ],

    "Bootstrap_Interpretation":
        (
            "Hierarchical bootstrap estimates showed "
            "descriptively higher Dark-domain AUC and "
            "lower Dark-domain Brier error, but these "
            "were treated as supportive uncertainty "
            "estimates rather than confirmatory evidence."
        ),

    "Fitzpatrick_Analysis":
        "Individual Fitzpatrick I-VI",

    "Fitzpatrick_Omnibus_Test":
        "Friedman repeated-measures test",

    "Fitzpatrick_Omnibus_Multiplicity":
        "Holm across five predefined outcomes",

    "Fitzpatrick_Holm_Supported_Outcomes":
        [
            "Entropy"
        ],

    "Fitzpatrick_Entropy_Friedman_P":
        0.004500,

    "Fitzpatrick_Entropy_Holm_P":
        0.022499,

    "Entropy_Posthoc_Test":
        (
            "Exact paired two-sided Wilcoxon "
            "with Holm correction across 15 pairs"
        ),

    "Entropy_Posthoc_Holm_Supported_Pairs":
        0,

    "Highest_Mean_Entropy_Fitzpatrick":
        6,

    "Highest_Mean_Entropy":
        0.277605,

    "Lowest_Mean_Entropy_Fitzpatrick":
        4,

    "Lowest_Mean_Entropy":
        0.187964,

    "Fitzpatrick_VI_N":
        25,

    "Fitzpatrick_VI_Benign":
        11,

    "Fitzpatrick_VI_Malignant":
        14,

    "Primary_Conclusion":
        (
            "No predefined Fair-Dark fairness outcome "
            "remained statistically supported after Holm "
            "correction across the five independent model seeds."
        ),

    "Secondary_Conclusion":
        (
            "Predictive entropy differed across Fitzpatrick "
            "I-VI at the omnibus level after Holm correction, "
            "but no individual pair remained statistically "
            "supported in corrected post-hoc testing."
        ),

    "Interpretive_Caution":
        (
            "Fitzpatrick VI showed the highest descriptive "
            "predictive entropy but contained only 25 test "
            "images, so this finding requires cautious "
            "interpretation."
        ),

    "Status":
        "COMPLETE",

    "Frozen":
        True
}


PHASE7_COMPLETION_FILE = (

    PHASE7_MANIFEST_PATH
    / "Phase7_Completion_Manifest.json"
)


with open(
    PHASE7_COMPLETION_FILE,
    "w"
) as f:

    json.dump(
        phase7_completion_manifest,
        f,
        indent=4
    )


print(
    "✓ Phase 7 completion manifest saved:"
)

print(
    PHASE7_COMPLETION_FILE
)

In [ ]:
# ============================================
# Section 13.3
# Final Phase 7 Hard Completion Gate
# ============================================

assert (
    PHASE7_COMPLETION_FILE.exists()
)


with open(
    PHASE7_COMPLETION_FILE,
    "r"
) as f:

    final_phase7_manifest = json.load(
        f
    )


assert (
    final_phase7_manifest[
        "Status"
    ]
    ==
    "COMPLETE"
)


assert (
    final_phase7_manifest[
        "Frozen"
    ]
    is True
)


assert (
    final_phase7_manifest[
        "Selected_Lambda"
    ]
    ==
    0.0
)


assert (
    final_phase7_manifest[
        "Model_Retraining_Performed"
    ]
    is False
)


assert (
    final_phase7_manifest[
        "Model_Selection_Reopened"
    ]
    is False
)


assert (
    final_phase7_manifest[
        "Primary_Number_Holm_Supported_Outcomes"
    ]
    ==
    0
)


assert (
    final_phase7_manifest[
        "Fitzpatrick_Holm_Supported_Outcomes"
    ]
    ==
    [
        "Entropy"
    ]
)


assert (
    final_phase7_manifest[
        "Entropy_Posthoc_Holm_Supported_Pairs"
    ]
    ==
    0
)


print(
    "✓ Phase 7 status: COMPLETE"
)

print(
    "✓ Phase 7 frozen: TRUE"
)

print(
    "✓ Authoritative model remains lambda=0."
)

print(
    "✓ No Phase 7 retraining occurred."
)

print(
    "✓ Fair-Dark Holm-supported outcomes: 0"
)

print(
    "✓ Fitzpatrick Holm-supported omnibus outcomes: 1"
)

print(
    "✓ Supported Fitzpatrick outcome: Entropy"
)

print(
    "✓ Entropy post-hoc supported pairs: 0"
)

print(
    "✓ No further Phase 7 model selection is permitted."
)